# Superior Ensemble Training for Paperspace
## Automated One-Click Training Pipeline

This notebook executes the complete training pipeline using the superior ensemble trainer system. Just run all cells to train models automatically.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

print("🔧 Setting up environment...")
project_root = Path("/notebooks/bot") if Path("/notebooks/bot").exists() else Path.cwd()
os.chdir(project_root)
sys.path.insert(0, str(project_root))

print(f"📁 Working directory: {project_root}")

# Pull latest changes from GitHub
print("🔄 Pulling latest changes from GitHub...")
try:
    result = subprocess.run(
        ["git", "pull", "--ff-only"], 
        capture_output=True, 
        text=True, 
        cwd=project_root,
        timeout=30
    )
    
    if result.returncode == 0:
        print("✅ Successfully pulled latest changes")
        if result.stdout.strip():
            print(f"   {result.stdout.strip()}")
    else:
        print("⚠️ Git pull failed, continuing with current code")
        if result.stderr:
            print(f"   Error: {result.stderr.strip()}")
        
except Exception as e:
    print(f"⚠️ Could not pull from git: {e}")
    print("   Continuing with current code...")

# Install required dependencies with timeout protection
print("📦 Installing critical dependencies...")

critical_packages = [
    "pandas", "numpy", "PyYAML", "python-dotenv",
    "ta", "yfinance", "torch", "lightgbm", "stable-baselines3"
]

for package in critical_packages:
    try:
        print(f"📦 Installing {package}...")
        result = subprocess.run([
            sys.executable, "-m", "pip", "install", package
        ], capture_output=True, text=True, timeout=60)
        
        if result.returncode == 0:
            print(f"✅ {package} installed successfully")
        else:
            print(f"⚠️ {package} installation failed, continuing...")
            
    except subprocess.TimeoutExpired:
        print(f"⏰ {package} installation timed out, continuing...")
    except KeyboardInterrupt:
        print(f"🛑 Installation interrupted at {package}")
        print("💡 You can continue with existing packages or restart")
        break
    except Exception as e:
        print(f"❌ {package} installation error: {e}")

# Install blinker separately to handle conflicts
try:
    print("📦 Installing blinker (conflict-prone package)...")
    result = subprocess.run([
        sys.executable, "-m", "pip", "install", "--ignore-installed", "blinker"
    ], capture_output=True, text=True, timeout=30)
    
    if result.returncode == 0:
        print("✅ blinker installed successfully")
    else:
        print("⚠️ blinker installation failed, continuing...")
        
except Exception as e:
    print(f"⚠️ blinker installation error: {e}")

# Verify critical packages
print("🔍 Verifying critical packages...")
verification_packages = {
    "pandas": "pandas",
    "numpy": "numpy", 
    "ta": "ta",
    "yfinance": "yfinance",
    "torch": "torch",
    "lightgbm": "lightgbm",
    "stable_baselines3": "stable_baselines3"
}

missing_packages = []
for name, import_name in verification_packages.items():
    try:
        __import__(import_name)
        print(f"✅ {name} - OK")
    except ImportError:
        print(f"❌ {name} - MISSING")
        missing_packages.append(name)

if missing_packages:
    print(f"⚠️ Missing packages: {missing_packages}")
    print("💡 Training may still work with available packages")
else:
    print("🎉 All critical packages verified!")

print(f"🐍 Python path: {sys.path[0]}")
print("🔧 Environment setup completed!")

In [ ]:
# Import and execute the superior training system
print("🚀 Starting Superior Ensemble Training...")

# First, verify critical dependencies are available
print("🔍 Checking dependencies...")
try:
    import ta
    import torch
    import lightgbm
    import stable_baselines3
    import yfinance
    import pandas
    import numpy
    print("✅ All critical dependencies verified")
except ImportError as e:
    print(f"❌ Missing dependency: {e}")
    print("💡 Please run the previous cell to install dependencies")
    raise

try:
    # Import the training runner directly (not as subprocess to avoid argument parsing issues)
    from paperspace_mlops.paperspace_superior_training import PaperspaceTrainingRunner
    
    print("✅ Superior training system imported successfully")
    
    # Create runner instance
    runner = PaperspaceTrainingRunner()
    
    # Execute training with default parameters (all models, all symbols)
    print("🎯 Launching automated training pipeline...")
    result = runner.run_training()
    
    if result["status"] == "complete":
        print("🎉 Training completed successfully!")
        print(f"✅ Symbols trained: {result['symbols_trained']}")
        print(f"✅ Models trained: {result['models_trained']}")
        print(f"✅ S3 export status: {result['export_status']}")
    else:
        print(f"⚠️ Training completed with issues:")
        print(f"   Status: {result['status']}")
        if result.get('errors'):
            print(f"   Errors: {result['errors']}")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("📝 Falling back to direct execution...")
    
    # Fallback: run the training script directly but without arguments
    import subprocess
    result = subprocess.run([
        sys.executable, 
        "paperspace_mlops/paperspace_superior_training.py",
        "--quick-test"  # Use quick test mode to avoid long training
    ], capture_output=True, text=True)
    
    print("STDOUT:", result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)
    print(f"Return code: {result.returncode}")
    
except Exception as e:
    print(f"❌ Training failed: {e}")
    import traceback
    traceback.print_exc()
    raise

In [ ]:
# Post-training validation and S3 export
print("📊 Running post-training validation...")

try:
    # Check for trained models
    models_dir = Path("models")
    if models_dir.exists():
        model_files = list(models_dir.rglob("*.pt")) + list(models_dir.rglob("*.pkl")) + list(models_dir.rglob("*.zip"))
        print(f"✅ Found {len(model_files)} model files:")
        for model_file in model_files[:10]:  # Show first 10
            print(f"  📁 {model_file}")
        if len(model_files) > 10:
            print(f"  ... and {len(model_files) - 10} more")
    else:
        print("⚠️ No models directory found")
    
    # Optional: Export to S3 if credentials are available
    if os.getenv("AWS_ACCESS_KEY_ID") and os.getenv("AWS_SECRET_ACCESS_KEY"):
        print("☁️ AWS credentials found - exporting to S3...")
        try:
            import subprocess
            result = subprocess.run([
                sys.executable, 
                "paperspace_mlops/export_to_s3.py"
            ], capture_output=True, text=True)
            
            if result.returncode == 0:
                print("✅ Models exported to S3 successfully!")
            else:
                print(f"⚠️ S3 export failed: {result.stderr}")
        except Exception as e:
            print(f"⚠️ S3 export error: {e}")
    else:
        print("ℹ️ No AWS credentials - skipping S3 export")
    
    print("🎯 Training pipeline completed!")
    
except Exception as e:
    print(f"⚠️ Post-training validation error: {e}")
    # Don't raise - this is just validation